In [ ]:
%pyspark
# ================================================================
# Germany Air Pollution Analysis 2023 - Final Interactive Version
# Python only: no raw HTML code.
# Interactive charts use Plotly: hover, zoom, pan, legend click.
# P1 = PM10, P2 = PM2.5
# ================================================================

from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
)
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")
spark.catalog.clearCache()

try:
    import plotly.graph_objects as go
    import plotly.express as px
    import plotly.io as pio
    from plotly.subplots import make_subplots
except ImportError:
    raise ImportError(
        "This final version needs Plotly for interactive hover charts. "
        "Ask your lab/admin to install plotly in the PySpark environment."
    )

# If one renderer does not show in your environment, try changing this to:
# "notebook_connected", "iframe", or "browser".
pio.renderers.default = "notebook"

WHO = {"PM10": 45, "PM25": 15}
COL = {"PM10": "#D94F3D", "PM25": "#2563EB"}
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
weekday_order = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]

schema = StructType([
    StructField("sensor_id", IntegerType(), True),
    StructField("sensor_type", StringType(), True),
    StructField("location", IntegerType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lon", DoubleType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("P1", DoubleType(), True),
    StructField("durP1", DoubleType(), True),
    StructField("ratioP1", DoubleType(), True),
    StructField("P2", DoubleType(), True),
    StructField("durP2", DoubleType(), True),
    StructField("ratioP2", DoubleType(), True),
])

# ================================================================
# 1. Load and clean with Spark
# No cache() and no full count(): avoids Spark disk-space problems.
# ================================================================

raw = (
    spark.read
    .option("header", "true")
    .option("delimiter", ";")
    .schema(schema)
    .csv("hdfs://bdgtm:8020/pm_data/DE_2023-*_sds011.csv")
)

clean = (
    raw.select("sensor_id", "lat", "lon", "timestamp", "P1", "P2")
    .filter(F.col("timestamp").isNotNull())
    .filter(F.col("P1").isNotNull() & F.col("P2").isNotNull())
    .filter(F.col("P1").between(0, 500) & F.col("P2").between(0, 500))
    .withColumn("date", F.to_date("timestamp"))
    .withColumn("hour", F.hour("timestamp"))
    .withColumn("day_of_week", F.dayofweek("timestamp"))
    .withColumn("month", F.month("timestamp"))
)

print("Clean data ready. Small preview only:")
clean.limit(3).show(truncate=False)

# ================================================================
# 2. Aggregate with Spark, then convert small results to pandas
# ================================================================

daily = (
    clean.groupBy("date")
    .agg(F.avg("P1").alias("PM10"), F.avg("P2").alias("PM25"))
    .orderBy("date")
    .toPandas()
)

if daily.empty:
    raise ValueError("No rows loaded. Check the HDFS path and data files.")

daily["date"] = pd.to_datetime(daily["date"])
daily = daily.sort_values("date").set_index("date")
daily["PM10_7d"] = daily["PM10"].rolling(7, min_periods=3).mean()
daily["PM25_7d"] = daily["PM25"].rolling(7, min_periods=3).mean()
daily["PM10_30d"] = daily["PM10"].rolling(30, min_periods=7).mean()
daily["PM25_30d"] = daily["PM25"].rolling(30, min_periods=7).mean()

hourly = (
    clean.groupBy("hour")
    .agg(F.avg("P1").alias("PM10"), F.avg("P2").alias("PM25"))
    .orderBy("hour")
    .toPandas()
)

dow = (
    clean.groupBy("day_of_week")
    .agg(F.avg("P1").alias("PM10"), F.avg("P2").alias("PM25"))
    .orderBy("day_of_week")
    .toPandas()
)
dow["label"] = dow["day_of_week"].map({
    1: "Sun", 2: "Mon", 3: "Tue", 4: "Wed", 5: "Thu", 6: "Fri", 7: "Sat"
})

monthly = (
    clean.groupBy("month")
    .agg(F.avg("P1").alias("PM10"), F.avg("P2").alias("PM25"))
    .orderBy("month")
    .toPandas()
)
monthly["label"] = [month_labels[int(m) - 1] for m in monthly["month"]]

print("Aggregations ready: daily, hourly, weekday, monthly.")


In [ ]:
%pyspark
# ================================================================
# 3. Interactive daily trend plot
# Hover shows date and value. Legend items can be clicked on/off.
# ================================================================

fig_daily = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("PM10 Daily Trend", "PM2.5 Daily Trend")
)

for row, metric in [(1, "PM10"), (2, "PM25")]:
    fig_daily.add_trace(
        go.Scatter(
            x=daily.index,
            y=daily[metric],
            mode="lines",
            name=metric + " daily mean",
            line=dict(color=COL[metric], width=1.4),
            opacity=0.55,
            hovertemplate="Date: %{x|%Y-%m-%d}<br>" + metric + ": %{y:.2f} ug/m3<extra></extra>",
        ),
        row=row, col=1,
    )
    fig_daily.add_trace(
        go.Scatter(
            x=daily.index,
            y=daily[metric + "_7d"],
            mode="lines",
            name=metric + " 7-day average",
            line=dict(color="#F97316", width=2.5),
            hovertemplate="Date: %{x|%Y-%m-%d}<br>7-day avg: %{y:.2f} ug/m3<extra></extra>",
        ),
        row=row, col=1,
    )
    fig_daily.add_trace(
        go.Scatter(
            x=daily.index,
            y=daily[metric + "_30d"],
            mode="lines",
            name=metric + " 30-day average",
            line=dict(color="#111827", width=2.2, dash="dash"),
            hovertemplate="Date: %{x|%Y-%m-%d}<br>30-day avg: %{y:.2f} ug/m3<extra></extra>",
        ),
        row=row, col=1,
    )
    fig_daily.add_hline(
        y=WHO[metric],
        line_dash="dot",
        line_color="red",
        annotation_text="WHO limit " + str(WHO[metric]) + " ug/m3",
        row=row, col=1,
    )

fig_daily.update_layout(
    title="Interactive Daily PM Concentrations with Moving Averages - Germany 2023",
    height=760,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_daily.update_yaxes(title_text="PM10 (ug/m3)", row=1, col=1)
fig_daily.update_yaxes(title_text="PM2.5 (ug/m3)", row=2, col=1)
fig_daily.update_xaxes(rangeslider_visible=True, row=2, col=1)
fig_daily.show()


In [ ]:
%pyspark
# ================================================================
# 4. Interactive seasonal patterns
# Hover over each point/bar to see exact values.
# ================================================================

fig_season = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Average by Hour", "Average by Day of Week", "Average by Month"),
    specs=[[{"type": "xy"}, {"type": "xy"}, {"type": "xy"}]]
)

for metric in ["PM10", "PM25"]:
    fig_season.add_trace(
        go.Scatter(
            x=hourly["hour"], y=hourly[metric],
            mode="lines+markers", name=metric + " hourly",
            line=dict(color=COL[metric], width=2),
            hovertemplate="Hour: %{x}:00<br>" + metric + ": %{y:.2f} ug/m3<extra></extra>",
        ),
        row=1, col=1,
    )
    fig_season.add_trace(
        go.Bar(
            x=dow["label"], y=dow[metric],
            name=metric + " weekday",
            marker_color=COL[metric], opacity=0.72,
            hovertemplate="Day: %{x}<br>" + metric + ": %{y:.2f} ug/m3<extra></extra>",
        ),
        row=1, col=2,
    )
    fig_season.add_trace(
        go.Scatter(
            x=monthly["label"], y=monthly[metric],
            mode="lines+markers", name=metric + " monthly",
            line=dict(color=COL[metric], width=2),
            hovertemplate="Month: %{x}<br>" + metric + ": %{y:.2f} ug/m3<extra></extra>",
        ),
        row=1, col=3,
    )

fig_season.update_layout(
    title="Interactive Seasonal Patterns - Hour, Weekday, Month",
    height=500,
    template="plotly_white",
    barmode="group",
    hovermode="closest",
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="left", x=0),
)
fig_season.update_xaxes(title_text="Hour", row=1, col=1)
fig_season.update_xaxes(title_text="Day", row=1, col=2)
fig_season.update_xaxes(title_text="Month", row=1, col=3)
fig_season.update_yaxes(title_text="ug/m3", row=1, col=1)
fig_season.show()


In [ ]:
%pyspark
# ================================================================
# 5. Interactive outlier detection
# Main method: IQR. Comparison method: z-score.
# ================================================================

# Presentation notes:
# Point outlier: one unusual value/day.
# Contextual outlier: unusual for its season, hour, or location.
# Collective outlier: a group of unusual values, such as a week-long pollution episode.
# Other methods: rolling z-score, DBSCAN, Isolation Forest.
# If not removing outliers: winsorise, use median, log-transform, or robust models.

def flag_iqr(series, k=1.5):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lo = q1 - k * iqr
    hi = q3 + k * iqr
    return (series < lo) | (series > hi), lo, hi

def flag_zscore(series, threshold=3.0):
    sd = series.std()
    if sd == 0 or pd.isna(sd):
        return pd.Series(False, index=series.index)
    z = (series - series.mean()) / sd
    return z.abs() > threshold

daily["PM10_iqr"], pm10_lo, pm10_hi = flag_iqr(daily["PM10"])
daily["PM25_iqr"], pm25_lo, pm25_hi = flag_iqr(daily["PM25"])
daily["PM10_z"] = flag_zscore(daily["PM10"])
daily["PM25_z"] = flag_zscore(daily["PM25"])
daily["PM10_winsor"] = daily["PM10"].clip(lower=pm10_lo, upper=pm10_hi)
daily["PM25_winsor"] = daily["PM25"].clip(lower=pm25_lo, upper=pm25_hi)

fig_out = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("PM10 IQR Outliers", "PM2.5 IQR Outliers")
)

for row, metric, lo, hi in [
    (1, "PM10", pm10_lo, pm10_hi),
    (2, "PM25", pm25_lo, pm25_hi),
]:
    flag = metric + "_iqr"
    fig_out.add_trace(
        go.Scatter(
            x=daily.index, y=daily[metric],
            mode="lines", name=metric + " daily mean",
            line=dict(color=COL[metric], width=1.6),
            hovertemplate="Date: %{x|%Y-%m-%d}<br>" + metric + ": %{y:.2f} ug/m3<extra></extra>",
        ),
        row=row, col=1,
    )
    outlier_df = daily[daily[flag]]
    fig_out.add_trace(
        go.Scatter(
            x=outlier_df.index, y=outlier_df[metric],
            mode="markers", name=metric + " IQR outliers",
            marker=dict(color="red", size=9, symbol="x"),
            hovertemplate="OUTLIER<br>Date: %{x|%Y-%m-%d}<br>" + metric + ": %{y:.2f} ug/m3<extra></extra>",
        ),
        row=row, col=1,
    )
    fig_out.add_hline(y=hi, line_dash="dot", line_color="red",
                      annotation_text="upper IQR fence", row=row, col=1)
    fig_out.add_hline(y=lo, line_dash="dot", line_color="green",
                      annotation_text="lower IQR fence", row=row, col=1)

fig_out.update_layout(
    title="Interactive Outlier Detection using IQR",
    height=720,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_out.update_xaxes(rangeslider_visible=True, row=2, col=1)
fig_out.show()

print("PM10 IQR outliers:", int(daily["PM10_iqr"].sum()),
      "| z-score outliers:", int(daily["PM10_z"].sum()))
print("PM2.5 IQR outliers:", int(daily["PM25_iqr"].sum()),
      "| z-score outliers:", int(daily["PM25_z"].sum()))


In [ ]:
%pyspark
# ================================================================
# 6. Interactive air-quality calendar
# Each cell is a day. Hover to see date, PM2.5, and class.
# ================================================================

AQI_C = ["#27ae60", "#f39c12", "#e67e22", "#e74c3c"]
AQI_L = ["Good (<10)", "Moderate (10-15)", "Sensitive (15-25)", "Unhealthy (>25)"]

def classify_pm25(v):
    if v < 10:
        return 0
    if v < 15:
        return 1
    if v < 25:
        return 2
    return 3

daily["AQI"] = daily["PM25"].apply(classify_pm25)
daily["AQI_label"] = daily["AQI"].apply(lambda x: AQI_L[int(x)])
cal_df = daily.reset_index().copy()
cal_df["week"] = ((cal_df["date"] - cal_df["date"].min()).dt.days // 7).astype(int)
cal_df["weekday"] = cal_df["date"].dt.day_name().str[:3]
cal_df["weekday_num"] = cal_df["date"].dt.weekday
cal_df["date_str"] = cal_df["date"].dt.strftime("%Y-%m-%d")

pivot_val = cal_df.pivot(index="weekday_num", columns="week", values="AQI")
pivot_pm = cal_df.pivot(index="weekday_num", columns="week", values="PM25")
pivot_date = cal_df.pivot(index="weekday_num", columns="week", values="date_str")
pivot_label = cal_df.pivot(index="weekday_num", columns="week", values="AQI_label")

custom = np.dstack([
    pivot_date.fillna("").values,
    pivot_pm.round(2).fillna(0).astype(str).values,
    pivot_label.fillna("").values,
])

fig_cal = go.Figure(
    data=go.Heatmap(
        z=pivot_val.values,
        x=pivot_val.columns,
        y=["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
        colorscale=[
            [0.00, AQI_C[0]], [0.24, AQI_C[0]],
            [0.25, AQI_C[1]], [0.49, AQI_C[1]],
            [0.50, AQI_C[2]], [0.74, AQI_C[2]],
            [0.75, AQI_C[3]], [1.00, AQI_C[3]],
        ],
        zmin=0, zmax=3,
        customdata=custom,
        hovertemplate="Date: %{customdata[0]}<br>PM2.5: %{customdata[1]} ug/m3<br>Class: %{customdata[2]}<extra></extra>",
        colorbar=dict(
            title="Class",
            tickvals=[0, 1, 2, 3],
            ticktext=AQI_L,
        ),
    )
)

fig_cal.update_layout(
    title="Interactive Air-Quality Calendar - Daily PM2.5 Classes",
    template="plotly_white",
    height=430,
    xaxis_title="Week of year",
    yaxis_title="Day of week",
)
fig_cal.show()


In [ ]:
%pyspark
# ================================================================
# 7. Interactive regional hotspots
# Hover over each sensor to see location, reading count, mean, and std.
# ================================================================

risk = (
    clean.filter(F.col("lat").between(47, 55) & F.col("lon").between(6, 15))
    .groupBy("sensor_id", "lat", "lon")
    .agg(
        F.count("*").alias("n"),
        F.round(F.avg("P2"), 2).alias("PM25_mean"),
        F.round(F.stddev("P2"), 2).alias("PM25_std")
    )
    .filter(F.col("n") > 100)
    .orderBy(F.desc("PM25_mean"))
    .toPandas()
)

if risk.empty:
    print("No regional risk rows found after filtering.")
else:
    fig_hot = px.scatter(
        risk,
        x="lon", y="lat",
        color="PM25_mean",
        size="n",
        hover_data={
            "sensor_id": True,
            "lat": ":.4f",
            "lon": ":.4f",
            "n": True,
            "PM25_mean": ":.2f",
            "PM25_std": ":.2f",
        },
        color_continuous_scale="YlOrRd",
        title="Interactive Spatial Hotspots - Sensor-Level PM2.5 Risk",
        labels={
            "lon": "Longitude",
            "lat": "Latitude",
            "PM25_mean": "Mean PM2.5",
            "n": "Readings",
        },
        height=650,
    )
    fig_hot.add_hline(y=47, line_width=0)
    fig_hot.update_layout(template="plotly_white")
    fig_hot.update_xaxes(range=[5.8, 15.2])
    fig_hot.update_yaxes(range=[47.0, 55.5], scaleanchor="x", scaleratio=1)
    fig_hot.show()

    top15 = risk.head(15).copy()
    top15["location"] = top15.apply(
        lambda r: str(round(r["lat"], 2)) + "N, " + str(round(r["lon"], 2)) + "E",
        axis=1,
    )
    fig_rank = px.bar(
        top15.sort_values("PM25_mean"),
        x="PM25_mean", y="location",
        orientation="h",
        color="PM25_mean",
        color_continuous_scale="YlOrRd",
        hover_data={"sensor_id": True, "n": True, "PM25_std": True},
        title="Top 15 Highest-Risk Sensor Locations",
        labels={"PM25_mean": "Annual mean PM2.5 (ug/m3)", "location": "Location"},
        height=580,
    )
    fig_rank.add_vline(x=WHO["PM25"], line_dash="dash", line_color="red",
                       annotation_text="WHO PM2.5 limit")
    fig_rank.update_layout(template="plotly_white")
    fig_rank.show()


In [ ]:
%pyspark
# ================================================================
# 8b. Interactive correlation heatmap
# Shows linear relationships between PM10, PM2.5, time-of-day,
# and month. Strong correlations guide where to target interventions.
# ================================================================

# Re-aggregate with hour+month per row so we can compute Pearson r
feat_pdf = (
    clean.groupBy("date", "hour", "month")
    .agg(F.avg("P1").alias("PM10"), F.avg("P2").alias("PM25"))
    .toPandas()
)

corr = feat_pdf[["PM10", "PM25", "hour", "month"]].corr()
labels = corr.columns.tolist()

fig_corr = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=labels,
    y=labels,
    colorscale="RdBu",
    zmid=0, zmin=-1, zmax=1,
    text=[[f"{v:.2f}" for v in row] for row in corr.values],
    texttemplate="%{text}",
    hovertemplate="<b>%{y} vs %{x}</b><br>r = %{z:.3f}<extra></extra>",
))
fig_corr.update_layout(
    title="Correlation Heatmap — PM10, PM2.5, Hour, Month",
    height=450,
    template="plotly_white",
)
fig_corr.show()

pm_r = corr.loc["PM10", "PM25"]
hour_r = corr.loc["PM25", "hour"]
month_r = corr.loc["PM25", "month"]
print(f"PM10 <-> PM2.5 correlation: r = {pm_r:.3f} (shared sources)")
print(f"PM2.5 <-> hour:             r = {hour_r:.3f} (traffic/heating cycles)")
print(f"PM2.5 <-> month:            r = {month_r:.3f} (seasonal heating)")


In [ ]:
%pyspark
# ================================================================
# 8c. PM2.5 30-day forecast
# Primary: Facebook Prophet (auto-detects weekly + yearly seasonality).
# Fallback: 30-day moving average baseline if Prophet is not installed.
# ================================================================

fc_df = (
    daily.reset_index()[["date", "PM25"]]
    .rename(columns={"date": "ds", "PM25": "y"})
    .dropna()
)

try:
    from prophet import Prophet

    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=False,
        interval_width=0.90,
    )
    m.fit(fc_df)
    future = m.make_future_dataframe(periods=30)
    forecast = m.predict(future)
    method = "Prophet"

    fig_fc = go.Figure()
    fig_fc.add_trace(go.Scatter(
        x=fc_df["ds"], y=fc_df["y"],
        mode="lines", name="Observed PM2.5",
        line=dict(color="#2563EB", width=1.5),
        hovertemplate="Date: %{x|%Y-%m-%d}<br>PM2.5: %{y:.2f} ug/m3<extra></extra>",
    ))
    fig_fc.add_trace(go.Scatter(
        x=forecast["ds"], y=forecast["yhat"],
        mode="lines", name="Prophet forecast",
        line=dict(color="#f97316", width=2.5),
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Forecast: %{y:.2f} ug/m3<extra></extra>",
    ))
    fig_fc.add_trace(go.Scatter(
        x=pd.concat([forecast["ds"], forecast["ds"][::-1]]),
        y=pd.concat([forecast["yhat_upper"], forecast["yhat_lower"][::-1]]),
        fill="toself",
        fillcolor="rgba(249,115,22,0.12)",
        line=dict(color="rgba(0,0,0,0)"),
        name="90% confidence band",
        hoverinfo="skip",
    ))

    next_30 = forecast[forecast["ds"] > fc_df["ds"].max()].head(30)
    exceed = int((next_30["yhat"] > WHO["PM25"]).sum())
    peak_val = float(next_30["yhat"].max())
    peak_day = next_30.loc[next_30["yhat"].idxmax(), "ds"].date()

except ImportError:
    # Baseline: carry forward 30-day rolling mean
    baseline = float(daily["PM25"].rolling(30, min_periods=7).mean().dropna().iloc[-1])
    last_date = daily.index[-1]
    future_dates = pd.date_range(last_date + pd.Timedelta("1d"), periods=30)
    method = "30-day moving average baseline"

    fig_fc = go.Figure()
    fig_fc.add_trace(go.Scatter(
        x=daily.index, y=daily["PM25"],
        mode="lines", name="Observed PM2.5",
        line=dict(color="#2563EB", width=1.5),
        hovertemplate="Date: %{x|%Y-%m-%d}<br>PM2.5: %{y:.2f} ug/m3<extra></extra>",
    ))
    fig_fc.add_trace(go.Scatter(
        x=future_dates,
        y=[baseline] * 30,
        mode="lines",
        name="Baseline forecast (30d MA)",
        line=dict(color="#f97316", width=2.5, dash="dot"),
        hovertemplate="Date: %{x|%Y-%m-%d}<br>Forecast: %{y:.2f} ug/m3<extra></extra>",
    ))
    exceed = int(baseline > WHO["PM25"])
    peak_val = baseline
    peak_day = future_dates[-1].date()

fig_fc.add_hline(
    y=WHO["PM25"], line_dash="dot", line_color="red",
    annotation_text="WHO PM2.5 limit 15 ug/m3",
)
fig_fc.update_layout(
    title=f"Interactive PM2.5 30-Day Forecast — {method}",
    height=520,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_fc.update_xaxes(rangeslider_visible=True)
fig_fc.show()

print(f"Method: {method}")
print(f"Forecast days exceeding WHO PM2.5 limit: {exceed}/30")
print(f"Peak forecast: {peak_val:.2f} ug/m3")


In [ ]:
%pyspark
# ================================================================
# 8. Interactive trend, conclusion, and action plan
# ================================================================

def make_trend_stats(series):
    s = series.dropna()
    if len(s) < 2:
        raise ValueError("Need at least 2 daily rows to fit a trend line.")
    x = np.arange(len(s), dtype=float)
    y = s.values.astype(float)
    slope, intercept = np.polyfit(x, y, 1)
    fitted = intercept + slope * x
    total_var = np.sum((y - y.mean()) ** 2)
    r2 = 0.0 if total_var == 0 else 1 - np.sum((y - fitted) ** 2) / total_var
    return {
        "dates": s.index,
        "values": y,
        "fitted": fitted,
        "slope": float(slope),
        "r2": float(r2),
    }

stats = {
    "PM10": make_trend_stats(daily["PM10"]),
    "PM25": make_trend_stats(daily["PM25"]),
}

fig_trend = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("PM10 Linear Trend", "PM2.5 Linear Trend")
)

for row, metric in [(1, "PM10"), (2, "PM25")]:
    st = stats[metric]
    fig_trend.add_trace(
        go.Scatter(
            x=st["dates"], y=st["values"],
            mode="lines", name=metric + " daily",
            line=dict(color=COL[metric], width=1.3), opacity=0.55,
            hovertemplate="Date: %{x|%Y-%m-%d}<br>" + metric + ": %{y:.2f} ug/m3<extra></extra>",
        ),
        row=row, col=1,
    )
    fig_trend.add_trace(
        go.Scatter(
            x=st["dates"], y=st["fitted"],
            mode="lines", name=metric + " trend line",
            line=dict(color="#111827", width=2.5),
            hovertemplate="Trend value: %{y:.2f} ug/m3<extra></extra>",
        ),
        row=row, col=1,
    )
    fig_trend.add_hline(
        y=WHO[metric], line_dash="dot", line_color="red",
        annotation_text="WHO limit", row=row, col=1,
    )

fig_trend.update_layout(
    title="Interactive Linear Trend Lines",
    height=720,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_trend.update_xaxes(rangeslider_visible=True, row=2, col=1)
fig_trend.show()

# Useful action-oriented results
worst_month_num = int(monthly.sort_values("PM25", ascending=False).iloc[0]["month"])
worst_month_name = month_labels[worst_month_num - 1]
worst_hour = int(hourly.sort_values("PM25", ascending=False).iloc[0]["hour"])
worst_weekday = dow.sort_values("PM25", ascending=False).iloc[0]["label"]
worst_day = daily.sort_values("PM25", ascending=False).index[0]
worst_day_value = float(daily.sort_values("PM25", ascending=False).iloc[0]["PM25"])
good_days = int((daily["AQI"] <= 1).sum())
bad_days = int((daily["AQI"] >= 2).sum())
trend_dir = "improving" if stats["PM25"]["slope"] < 0 else "worsening"

if risk.empty:
    exceed_count = 0
    hotspot_text = "No reliable hotspot summary available after filtering."
else:
    exceed_count = int((risk["PM25_mean"] > WHO["PM25"]).sum())
    hotspot_text = str(exceed_count) + "/" + str(len(risk)) + " sensor locations exceed the WHO PM2.5 limit"

# Final interactive dashboard with KPI cards and important plots
indicator_specs = [[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}],
                   [{"type": "xy", "colspan": 2}, None, {"type": "xy", "colspan": 2}, None]]
fig_dash = make_subplots(
    rows=2, cols=4,
    specs=indicator_specs,
    subplot_titles=("Worst Month", "Worst Hour", "Worst Day", "PM2.5 Trend",
                    "Monthly PM2.5 Risk", "", "Hourly PM2.5 Pattern", ""),
    vertical_spacing=0.22,
    horizontal_spacing=0.08,
)

fig_dash.add_trace(go.Indicator(mode="number", value=worst_month_num,
                                title={"text": worst_month_name}), row=1, col=1)
fig_dash.add_trace(go.Indicator(mode="number", value=worst_hour,
                                title={"text": "hour of day"}), row=1, col=2)
fig_dash.add_trace(go.Indicator(mode="number", value=worst_day_value,
                                number={"suffix": " ug/m3"},
                                title={"text": str(worst_day.date())}), row=1, col=3)
fig_dash.add_trace(go.Indicator(mode="number", value=stats["PM25"]["slope"],
                                number={"valueformat": ".4f", "suffix": " /day"},
                                title={"text": trend_dir}), row=1, col=4)

fig_dash.add_trace(go.Bar(x=monthly["label"], y=monthly["PM25"],
                          marker_color="#2563EB",
                          name="Monthly PM2.5",
                          hovertemplate="Month: %{x}<br>PM2.5: %{y:.2f} ug/m3<extra></extra>"),
                   row=2, col=1)
fig_dash.add_trace(go.Scatter(x=hourly["hour"], y=hourly["PM25"],
                              mode="lines+markers", name="Hourly PM2.5",
                              line=dict(color="#D94F3D", width=3),
                              hovertemplate="Hour: %{x}:00<br>PM2.5: %{y:.2f} ug/m3<extra></extra>"),
                   row=2, col=3)
fig_dash.add_hline(y=WHO["PM25"], line_dash="dot", line_color="red", row=2, col=1)

fig_dash.update_layout(
    title="Final Interactive Action Dashboard - Germany PM2.5 Risk 2023",
    height=760,
    template="plotly_white",
    showlegend=False,
)
fig_dash.update_yaxes(title_text="PM2.5 (ug/m3)", row=2, col=1)
fig_dash.update_yaxes(title_text="PM2.5 (ug/m3)", row=2, col=3)
fig_dash.update_xaxes(title_text="Month", row=2, col=1)
fig_dash.update_xaxes(title_text="Hour", row=2, col=3)
fig_dash.show()

print("")
print("=" * 72)
print("CONCLUSIONS AND ACTION PLAN - Germany Air Quality 2023")
print("=" * 72)
print("1. PM2.5 trend is " + trend_dir +
      " with slope " + str(round(stats["PM25"]["slope"], 4)) + " ug/m3/day.")
print("2. R2 = " + str(round(stats["PM25"]["r2"], 3)) +
      ", so seasonality and events matter more than a simple straight line.")
print("3. Worst PM2.5 month: " + worst_month_name + ".")
print("4. Worst PM2.5 hour: " + str(worst_hour) + ":00.")
print("5. Worst PM2.5 weekday: " + str(worst_weekday) + ".")
print("6. Worst PM2.5 day: " + str(worst_day.date()) +
      " with " + str(round(worst_day_value, 2)) + " ug/m3.")
print("7. PM2.5 IQR outlier days: " + str(int(daily["PM25_iqr"].sum())) +
      "; z-score outlier days: " + str(int(daily["PM25_z"].sum())) + ".")
print("8. Good/moderate days: " + str(good_days) + "/" + str(len(daily)) +
      "; at-risk days: " + str(bad_days) + "/" + str(len(daily)) + ".")
print("9. " + hotspot_text + ".")
print("10. Recommended action: focus warnings, monitoring, traffic/emission checks")
print("    on the worst month, worst hour, outlier days, and high-risk sensor areas.")
print("=" * 72)
